In [1]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# # dx = 1 km; Np = 1M; Nt = 5 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
# t_res='5min'; res='1km'
# res='1km'
# Np_str='1e6'

# # dx = 1 km; Np = 1M; Nt = 1 min
# data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6_5min.nc') #***
# parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6_5min.nc') #***
# t_res='5min'; res='1km'
# Np_str='1e6'

# dx = 1km; Np = 50M
#Importing Model Data
check=False
dir2='/home/air673/koa_scratch/'
data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'


# # dx = 250 m
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***
# res='250m'
# Np_str='150e6'

In [2]:
def check_memory():
    import sys
    ipython_vars = ["In", "Out", "exit", "quit", "get_ipython", "ipython_vars"]
    print("Top 10 objects with highest memory usage")
    # Get a sorted list of the objects and their sizes
    mem = {
        key: round(value/1e6,2)
        for key, value in sorted(
            [
                (x, sys.getsizeof(globals().get(x)))
                for x in globals()
                if not x.startswith("_") and x not in sys.modules and x not in ipython_vars
            ],
            key=lambda x: x[1],
            reverse=True)[:10]
    }
    print({key:f"{value} MB" for key,value in mem.items()})
    print(f"\n{round(sum(mem.values()),2)/1000} GB in use overall")

In [4]:
###########################################################################################################################################################################

In [5]:
#LOADING VARIABLES
###############################################################

In [4]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
open_file=dir2+f'lagrangian_binary_array_{res}_{Np_str}_5min.h5'
with h5py.File(open_file, 'r') as f:
    Z = f['Z'][:]
    Y = f['Y'][:]
    X = f['X'][:]

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)
check_memory()

Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'NamespaceMagics': '0.0 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'times': '0.0 MB', 'open_file': '0.0 MB', 'open': '0.0 MB'}

3.192 GB in use overall


In [ ]:
##################################
#CALCULATION

In [5]:
#constants
Cp=1004 #Jkg-1K-1
Cv=717 #Jkg-1K-1
Rd=Cp-Cv #Jkg-1K-1
eps=0.608

Lx=(data['xf'][-1].item()-data['xf'][0].item())*1000 #x length (m)
Ly=(data['yf'][-1].item()-data['yf'][0].item())*1000 #y length (m)
Np=len(parcel['xh']) #number of lagrangian parcles
dt=(data['time'][1]-data['time'][0]).item()/1e9 #sec
dx=(data['xf'][1].item()-data['xf'][0].item())*1e3 #meters
dy=(data['yf'][1].item()-data['yf'][0].item())*1e3 #meters
xs=data['xf'].values*1000
ys=data['yf'].values*1000
zs=data['zf'].values*1000

def zf(z):
    k=z #z is the # level of z
    out=data['zf'].values[k]*1000
    return out
# def rho(x,y,z,t):
#     p=data['prs'].isel(xh=x,yh=y,zh=z,time=t).item()
#     p0=101325 #Pa
#     theta=data['th'].isel(xh=x,yh=y,zh=z,time=t).item()
#     T=theta*(p/p0)**(Rd/Cp)
#     qv=data['qv'].isel(xh=x,yh=y,zh=z,time=t).item()
#     # Tv=T*(1+eps*qv)
#     Tv=T*(eps+qv)/(eps*(1+qv))
#     rho = p/(Rd*Tv)
#     out=rho
#     return out

# rho_data=data['rho'].data
def rho(x,y,z,t):
    # out=data['rho'].isel(xh=x,yh=y,zh=z,time=t).item()
    out=rho_data[t,z,y,x]
    return out
def m(t):
    m=0
    #triple sum
    for k in range(len(data['zh'])):
        dz=(zf(k+1)-zf(k))
        for j in range(len(data['yh'])):
            for i in range(len(data['xh'])):
                rho_out=rho(i,j,k,t)
                m+=rho_out*dz
                
    #triple sum
    out=m*dx*dy/Np
    return out

In [6]:
#Calculate Mass Constant
# calculate='single_time'
calculate=False

dir3=dir+f'Project_Algorithms/Entrainment/'
if calculate==True:
    Nt=len(data['time'])
    m_arr=np.zeros((Nt))
    for t in np.arange(Nt):
        if np.mod(t,25)==0: print(t)
        # m_arr[t]=m(t) #UNCOMMENT FOR FULL CALCULATION
    np.save(dir3+'Mass_Array_5min.npy', m_arr)
    # np.save(dir3+'Mass_Array_1min.npy', m_arr)
elif calculate=='single_time':
    Nt=len(data['time'])
    m_arr=np.zeros((Nt))

    t=len(data['time'])//2
    m_300=m(t)
    for t in np.arange(Nt):
        m_arr[t]=m_300 #UNCOMMENT FOR FULL CALCULATION
    np.save(dir3+'Mass_Array_5min.npy', m_arr)
    # np.save(dir3+'Mass_Array_1min.npy', m_arr)
else:
    m_arr = np.load(dir3+'Mass_Array_5min.npy')
    # m_arr = np.load(dir3+'Mass_Array_1min.npy')

# # TESTING
# lst=[]
# for t in np.arange(133):
#     lst.append(m_arr[t])

# plt.plot(lst)
# (np.max(lst)-np.min(lst))*100/np.mean(lst)

In [7]:
def call_variable(varname):
    
    #READING BACK IN
    PROCESSING=False
    PROCESSING=True
    
    print('loading vars')
    if PROCESSING==False:
        dir3=dir+f'Project_Algorithms/Entrainment/3D_entrainmentdetrainment_profiles_{res}_{t_res}_{Np_str}.h5'
    if PROCESSING==True:
        dir3=dir+f'Project_Algorithms/Entrainment/3D_entrainmentdetrainment_profiles_PREPROCESSING_{res}_{t_res}_{Np_str}.h5'
    with h5py.File(dir3, "r") as h5f:
        if varname=='e_g':
            profile_array = h5f["profile_array_e_g"][:]
        elif varname=='e_c':
            profile_array = h5f["profile_array_e_c"][:]
        elif varname=='d_g':
            profile_array = h5f["profile_array_d_g"][:]
        elif varname=='d_c':
            profile_array = h5f["profile_array_d_c"][:]
    
    def apply_constant(profile_array,apply):
        if apply==True:
            Nt=profile_array.shape[0]
            Nz=profile_array.shape[1]
        
            profile_array/=(dx*dy*dt)
            for t in np.arange(Nt):
                profile_array[t]*=m_arr[t]
            for z in np.arange(Nz):
                dz=zf(z+1)-zf(z)
                profile_array[:,z]/=dz
        return profile_array
    
    #APPLY CONSTANTS TO ENTRAINMENT VALUE
    ##################################################
    
    profile_array=apply_constant(profile_array,apply=True)
    if varname in ['d_g','d_c']:
        profile_array*=-1
    ##################################################

    return profile_array

In [8]:
def make_lagrangian_array(varname):
    var_data=call_variable(varname)
    VAR=np.zeros_like(Z,dtype='float32')
    
    Nt=len(data['time'])
    Np=len(parcel['xh'])
    for p in np.arange(Np):
        if np.mod(p,2e5)==0: print(f"{p}/{len(parcel['xh'])}")
    
        #Get Indicies
        zs=Z[:,p]
        ys=Y[:,p]
        xs=X[:,p]
        ts = np.arange(Nt)  
    
        #Get Values
        vars = var_data[ts, zs, ys, xs]

        #Adding to Lagrangian Array
        VAR[:,p]=vars

        del vars
    del var_data
    return VAR

In [9]:
print('Working on Entrainment Variables')
E_G=make_lagrangian_array('e_g'); check_memory() 
E_C=make_lagrangian_array('e_c'); check_memory() 
D_G=make_lagrangian_array('d_g'); check_memory() 
D_C=make_lagrangian_array('d_c'); check_memory() 

Working on Entrainment Variables
loading vars
0/1000000
200000/1000000
400000/1000000
600000/1000000
800000/1000000
Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'E_G': '532.0 MB', 'xs': '0.0 MB', 'NamespaceMagics': '0.0 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB', 'times': '0.0 MB'}

3.724 GB in use overall
loading vars
0/1000000
200000/1000000
400000/1000000
600000/1000000
800000/1000000
Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'E_G': '532.0 MB', 'E_C': '532.0 MB', 'xs': '0.0 MB', 'NamespaceMagics': '0.0 MB', 'Normalize': '0.0 MB', 'MaxNLocator': '0.0 MB', 'ScalarFormatter': '0.0 MB'}

4.256 GB in use overall
loading vars
0/1000000
200000/1000000
400000/1000000
600000/1000000
800000/1000000
Top 10 objects with highest memory usage
{'Z': '1064.0 MB', 'Y': '1064.0 MB', 'X': '1064.0 MB', 'E_G': '532.0 MB', 'E_C': '532.0 MB', 'D_G': '532.0 MB', 'xs

In [ ]:
# Saving Data
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Arrays/'
out_file=dir2+f'ED_binary_array_{res}_{t_res}_{Np_str}.h5' 
with h5py.File(out_file, 'w') as f:
    # Save the array as a variable in the file
    f.create_dataset('E_G', data=E_G) 
    f.create_dataset('E_C', data=E_C)
    f.create_dataset('D_G', data=D_G)
    f.create_dataset('D_C', data=D_C)
print('done')

In [ ]:
#########################################

In [ ]:
# # Reading Back Data Later
# ##############
# import h5py
# dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
# with h5py.File(dir2+f'ED_binary_array_{res}_{Np_str}.h5', 'r') as f:
#     # Load the dataset by its name
#     E_G = f['E_G'][:]
#     E_C = f['E_C'][:]
#     D_G = f['D_G'][:]
#     D_C = f['D_C'][:]
# check_memory()